In [5]:
%pip install -q "qiskit==2.2.3" "qiskit-algorithms==0.4.0" "qiskit-nature==0.7.2" "pyscf==2.10.0"

import sys
from importlib.metadata import version
print("Python:", sys.version.split()[0])
for package in ["qiskit", "qiskit-algorithms", "qiskit-nature", "pyscf", "numpy", "scipy", "scikit-learn"]:
    print(package, version(package))
print("설치 완료: 다음 실행 코드 셀로 이동합니다.")

Python: 3.13.16
qiskit 2.2.3
qiskit-algorithms 0.4.0
qiskit-nature 0.7.2
pyscf 2.10.0
numpy 2.1.3
scipy 1.16.3
scikit-learn 1.6.1
설치 완료: 다음 실행 코드 셀로 이동합니다.


In [7]:
%matplotlib inline
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector


# ==============================================================================
# STEP 2. 소규모 데이터셋 생성 및 Train/Test 분할
# ==============================================================================
# 10건의 사용자 로그인 빈도와 이용 시간, 그리고 이탈 여부(churn)로 구성된 모의 데이터를 정의합니다.
# 훈련 데이터와 테스트 데이터로 분할합니다 (Stratified Split 적용).
data = pd.DataFrame({
    "login_freq": [3, 8, 1, 10, 6, 2, 9, 4, 7, 5],
    "usage_time": [12, 40, 5, 50, 35, 10, 45, 20, 30, 25],
    "churn": [1, 0, 1, 0, 0, 1, 0, 1, 0, 0]
})

X_train, X_test, y_train, y_test = train_test_split(
    data[["login_freq", "usage_time"]],
    data["churn"],
    test_size=0.3,
    random_state=42,
    stratify=data["churn"]
)


# ==============================================================================
# STEP 3. 데이터 전처리 (MinMaxScaler 스케일링)
# ==============================================================================
# Qiskit Feature Map에 입력하기 적합하도록 특성 값을 [0, π/2] 범위로 정규화합니다.
# 데이터 유출(Data Leakage)을 방지하기 위해 훈련 데이터에만 fit을 수행합니다.
scaler = MinMaxScaler(feature_range=(0, np.pi / 2), clip=True)
train = scaler.fit_transform(X_train)
test = scaler.transform(X_test)


# ==============================================================================
# STEP 4. 양자 피처 맵(Feature Map) 및 상태 변환 함수 정의
# ==============================================================================
# 2차원 특성을 양자 상태로 매핑하기 위해 ZZFeatureMap(reps=2)을 생성하고,
# 각 데이터 행을 양자 상태벡터(Statevector)로 변환하는 인코딩 함수를 정의합니다.
feature_map = zz_feature_map(feature_dimension=2, reps=2)

def encode(rows):
    """입력 데이터 행들을 받아 양자 상태벡터 배열로 변환합니다."""
    return np.array([
        Statevector.from_instruction(feature_map.assign_parameters(row)).data
        for row in rows
    ])

train_states = encode(train)
test_states = encode(test)


# ==============================================================================
# STEP 5. 양자 커널(Quantum Kernel) 계산
# ==============================================================================
# 양자 상태 간의 충실도(Fidelity, 내적의 절댓값 제곱)를 이용해
# 훈련 커널(K_train)과 테스트-훈련 간 커널(K_test)을 계산합니다.
K_train = np.abs(train_states.conj() @ train_states.T) ** 2
K_test = np.abs(test_states.conj() @ train_states.T) ** 2


# ==============================================================================
# STEP 6. 사전 계산된 커널(Precomputed Kernel)을 이용한 SVC 학습 및 예측
# ==============================================================================
# 커널 행렬을 직접 입력받는 SVC 모델을 생성하고, 훈련 커널로 학습한 뒤 테스트 커널로 예측합니다.
model = SVC(kernel="precomputed", C=1.0)
model.fit(K_train, y_train)

pred = model.predict(K_test)


# ==============================================================================
# STEP 7. 결과 출력 및 커널 수학적 성질 검증 (Assertions)
# ==============================================================================
print("=" * 60)
print("QSVM Simulation Results")
print("=" * 60)
print("훈련 커널 크기 :", K_train.shape, "/ 평가 커널 크기:", K_test.shape)
print("실제 이탈 여부 :", y_test.to_numpy())
print("예측 이탈 여부 :", pred)
print(f"테스트 정확도  : {accuracy_score(y_test, pred):.4f}")
print("혼동행렬(행: 실제 / 열: 예측):\n", confusion_matrix(y_test, pred, labels=[0, 1]))

# 커널 행렬의 수학적 성질 검증 (대칭성, 대각선 요소가 1인지, 준정치성 확인)
assert np.allclose(K_train, K_train.T), "커널 행렬은 대칭이어야 합니다!"
assert np.allclose(np.diag(K_train), 1), "커널 행렬의 대각 원소는 1이어야 합니다!"
assert np.linalg.eigvalsh(K_train).min() > -1e-8, "커널 행렬은 양의 준정치(PSD)여야 합니다!"
assert len(pred) == 3, "테스트 샘플 수와 예측 결과 수가 일치하지 않습니다!"

print("\n" + "=" * 60)
print("소규모 시뮬레이션 결과이며 양자 우위나 실무 예측 성능을 입증하지 않습니다.")
print("=" * 60)

QSVM Simulation Results
훈련 커널 크기 : (7, 7) / 평가 커널 크기: (3, 7)
실제 이탈 여부 : [0 0 1]
예측 이탈 여부 : [0 0 0]
테스트 정확도  : 0.6667
혼동행렬(행: 실제 / 열: 예측):
 [[2 0]
 [1 0]]

소규모 시뮬레이션 결과이며 양자 우위나 실무 예측 성능을 입증하지 않습니다.
